<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# BM25

**[BM25](https://learning.nextia-ai.com/courses/search/m03/bm25/)** · Search and Ranking: Find the Best Match · Module 3, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute BM25 by hand, explain each part of a real score, and tune field weights on the validation queries.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 10 MB) and the packages. No account, no key and no model download. |
| **You should know** | The inverted index, from [The inverted index](https://learning.nextia-ai.com/courses/search/m03/the-inverted-index/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m03/bm25/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M03-L02-bm25/bm25-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. BM25 by hand, on six products

BM25 gives each product a score for a query. Each query word adds a **part** to the score. The part is large when the word is **rare** in the catalogue (few products have it), when it appears **often** in the product, and when the product is **short**. Run the next cell: the parts for "outdoor hose" in six small products.

> **Predict.** "hose" is in 2 of the 6 products, and "outdoor" in 3. Which product comes first: C "garden hose with reel" or F "hose reel cart"? Why?

In [ ]:
import math, re

products = {
    "A": "grey outdoor cushion",
    "B": "blue outdoor cushion cover",
    "C": "garden hose with reel",
    "D": "grey sofa",
    "E": "outdoor sofa with cushions",
    "F": "hose reel cart",
}
def tokens(text):
    return [t for t in re.split(r"[^\w]+|_", str(text).lower()) if t]

K1, B = 1.2, 0.75
def idf(df, n):
    """Rarity: high for a word in few products."""
    return math.log(1 + (n - df + 0.5) / (df + 0.5))

def part(tf, df, n, length, avg):
    """The BM25 part of one query word in one product (0 when the word is not in it)."""
    if tf == 0:
        return 0.0
    return idf(df, n) * tf / (tf + K1 * (1 - B + B * length / avg))

toks = {p: tokens(t) for p, t in products.items()}
n = len(toks)
avg = sum(len(t) for t in toks.values()) / n
query = tokens("outdoor hose")
df = {w: sum(w in t for t in toks.values()) for w in query}
print(f"{n} products, average length {avg:.3f} words; df {df}")
hand = {}
for p, t in toks.items():
    parts = [part(t.count(w), df[w], n, len(t), avg) for w in query]
    hand[p] = sum(parts)
    print(p, f"{products[p]:28}", " + ".join(f"{x:.3f}" for x in parts), f"= {hand[p]:.3f}")

> **Check.** F comes first with 0.488, then C with 0.433. Both have "hose" once. F is shorter (3 words against 4), so its part is larger. The "hose" products beat the "outdoor" products, because "hose" is rarer.

## 2. The same numbers from a library

`bm25s` is the BM25 library of the course project. Run the next cell: it gives the same scores as your hand calculation.

In [ ]:
import bm25s
tiny = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
tiny.index(list(toks.values()), show_progress=False)
lib = tiny.get_scores(query)
for p, s in zip(toks, lib):
    print(p, f"by hand {hand[p]:.3f}   bm25s {s:.3f}")

## 3. BM25 on the whole catalogue

The next cell builds five BM25 indexes of the catalogue, one for each **field**: title and description together, title, description, category, and attributes. It takes about a minute. Then it searches "emma headboard".

In [ ]:
cat = catalogue()
ids = cat["product_id"].to_numpy()
FIELDS = {
    "all": cat["title"] + " " + cat["description"],
    "title": cat["title"],
    "description": cat["description"],
    "category": cat["product_class"] + " " + cat["category"],
    "attributes": cat["attributes"] + " " + cat["colour"] + " " + cat["style"] + " " + cat["material"],
}
indexes, lengths = {}, {}
for field, texts in FIELDS.items():
    corpus = [tokens(t) for t in texts]
    r = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
    r.index(corpus, show_progress=False)
    indexes[field] = r
    lengths[field] = np.array([len(c) for c in corpus])

def scores(query, field="all"):
    """The BM25 score of every product for a query, in one field."""
    r = indexes[field]
    words = [w for w in tokens(query) if w in r.vocab_dict]
    return r.get_scores(words) if words else np.zeros(len(ids))

def top(score, k=100):
    """The k best product IDs with a score above 0 (ties by product ID, as in the project)."""
    nz = np.flatnonzero(score > 0)
    nz = nz[np.lexsort((ids[nz], -score[nz]))][:k]
    return list(ids[nz])

print(f"{len(ids):,} products; average length (title and description) {lengths['all'].mean():.1f} words")
show(top(scores("emma headboard")), n=3)

A bird activity centre comes first for "emma headboard". The next cell explains why: it prints every part of the score for two products.

In [ ]:
row = {p: i for i, p in enumerate(ids)}
avg_all = lengths["all"].mean()
docs_all = [tokens(t) for t in FIELDS["all"]]
def explain(query, product):
    i = row[product]
    total = 0.0
    for w in tokens(query):
        df_w = int(np.count_nonzero(scores(w))) if w in indexes["all"].vocab_dict else 0
        tf = docs_all[i].count(w)
        x = part(tf, df_w, len(ids), lengths["all"][i], avg_all)
        total += x
        print(f"  {w:10} tf {tf}  df {df_w:5}  IDF {idf(df_w, len(ids)):.3f}  length {lengths['all'][i]:3}  part {x:.3f}")
    print(f"  sum {total:.3f}; bm25s {scores(query)[i]:.3f}")
for p in ["LK-97580", "LK-45931"]:
    print(p, cat.loc[row[p], "title"])
    explain("emma headboard", p)

> **Check.** "emma" is in only 6 products, so its IDF is high (8.727). The bird activity centre has only 7 words, so one "emma" is worth 6.288. The headboard has "headboard" 5 times, but "headboard" is in 1,089 products: its part is only about half as large.

## 4. Your turn: one part by hand

Compute the part of "headboard" in product LK-45931 with the function `part()`: tf 5, df 1,089, length 48 words, and the catalogue's average length `avg_all`.

In [ ]:
# "headboard" in LK-45931: tf 5, df 1,089, length 48 (use part() and avg_all)
headboard_part = ...
print(headboard_part)

In [ ]:
expect('headboard_part', headboard_part, np.float64(3.054), tolerance=0.001)

## 5. Fields with weights

A product has fields: a title, a description, a category, attributes. With one index per field, you can give each field a **weight**: its score is multiplied by the weight, and the weighted scores are added. Run the next cell. It measures the one-field index on the 121 validation queries.

In [ ]:
q = queries()
j = judgments()
val = q[q["split"] == "validation"]
test = q[q["split"] == "test"]

def weighted(query, weights):
    """The weighted sum of the BM25 scores of each field."""
    total = np.zeros(len(ids))
    for field, w in weights.items():
        if w:
            total += w * scores(query, field)
    return total

def ndcg(split, weights):
    run = {qid: top(weighted(text, weights)) for qid, text in zip(split["query_id"], split["query"])}
    return cp.per_query(run, j)["ndcg@10"].mean()

one_field = ndcg(val, {"all": 1})
print(f"one field (title and description): NDCG@10 {one_field:.3f} on {len(val)} validation queries")

> **Check.** One field gives NDCG@10 0.432 on validation, the same as `search eval --run bm25 --split validation`.

## 6. Your turn: tune the title weight on validation

Try the title weights 1, 2, 3 and 5, with description 1, category 0.5 and attributes 0. Measure each on the **validation** queries, never on the test queries. Keep the best one.

In [ ]:
results = {}
for t in [1, 2, 3, 5]:
    results[t] = ...   # ndcg(val, ...) with title t, description 1, category 0.5, attributes 0
    print(f"title {t}: NDCG@10 {results[t]}")
best_title = ...   # the title weight with the highest NDCG@10 on validation
print("best title weight on validation:", best_title)

In [ ]:
expect('best_title', best_title, 1)

In [ ]:
expect('NDCG@10 of the best title weight', results.get(best_title) if isinstance(results, dict) else None, np.float64(0.459), tolerance=0.002)

## 7. Now look at the test queries, once

Run the next cell. It measures your tuned weights and a hand-picked setting (title 3, description 1, category 1, attributes 0.5) on validation and on test.

> **Predict.** The tuned weights win on validation. Do they also win on test?

In [ ]:
if best_title is ...:
    print("Not yet: complete the task above first.")
else:
    tuned = {"title": best_title, "description": 1, "category": 0.5, "attributes": 0}
    picked = {"title": 3, "description": 1, "category": 1, "attributes": 0.5}
    for name, w in [("one field", {"all": 1}), ("tuned on validation", tuned), ("hand-picked 3, 1, 1, 0.5", picked)]:
        print(f"{name:26} validation {ndcg(val, w):.3f}   test {ndcg(test, w):.3f}")

> **Check.** The tuned weights win on validation by a small margin, and lose on test. The differences on validation are a few thousandths: that is noise for 121 queries. The procedure is still right: choose on validation, report test once, and do not go back to change the weights after you see test. The lesson explains why.

In the project: `search eval --run bm25-fields --split validation`, `search explain "emma headboard" LK-97580`, and `search eval --run bm25-fields --set title=3 --set category=1 --set attributes=0.5`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Exact matches and rules](https://learning.nextia-ai.com/courses/search/m03/exact-matches-and-rules/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m03/bm25/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.